# HAM10000 — Stage 4: Transfer Learning

**Goal:** Fine-tune ResNet18 and EfficientNet-B0 (ImageNet pretrained) on HAM10000.
Compare against the Stage 3 baseline CNN across accuracy, macro-F1, and malignant-class recall.

**Strategy: fine-tune end-to-end with two-speed learning rates**
- Backbone: `lr = 1e-4` — 10× lower to protect pretrained features (prevents catastrophic forgetting)
- New head: `lr = 1e-3` — trained at full speed (random init needs aggressive updates)
- Rationale: dermoscopic images differ enough from ImageNet that the backbone must adapt,
  not just the classifier head. Freezing the backbone would leave texture-sensitive
  features unexploited for lesion classification.

**Same across all models (fair comparison):**
- Same train/val/test splits (random_state=42)
- Same class-weighted CrossEntropyLoss
- Same `train()` loop, `evaluate_model()`, early stopping (patience=7)
- Same augmentation pipeline

## Cell 1 — Clone / update repo and set up paths

In [ ]:
import os, sys, subprocess

REPO_URL  = "https://github.com/Dev252001/HAM10000.git"
REPO_DIR  = "/content/ham10000-classifier"
SRC_DIR   = os.path.join(REPO_DIR, "src")
DATA_DIR  = os.path.join(REPO_DIR, "data")
OUT_DIR   = os.path.join(REPO_DIR, "outputs")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("Repo updated.")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"src/ on path: {SRC_DIR}")

## Cell 2 — Install dependencies

In [ ]:
!pip install -q \
  "numpy>=2.0" \
  "pandas>=2.2.2" \
  "Pillow>=10.4.0" \
  "scikit-learn>=1.5.0" \
  "matplotlib>=3.9.0" \
  "seaborn>=0.13.2" \
  "kaggle>=1.6.14" \
  "ipywidgets>=8.1.3"
print("Dependencies ready.")

## Cell 3 — Verify GPU

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — training will be very slow. Runtime → Change runtime type → T4 GPU.")

## Cell 4 — Dataset (Google Drive cache + Kaggle fallback)

Loads data from Drive if available (instant), otherwise downloads from Kaggle once and saves to Drive.

In [ ]:
import os, json, shutil
from google.colab import drive
from data_loader import download_dataset

DRIVE_DATA = "/content/drive/MyDrive/HAM10000_data"
DATA_DIR   = "/content/ham10000-classifier/data"

drive.mount('/content/drive')

if os.path.exists(os.path.join(DRIVE_DATA, 'HAM10000_metadata.csv')):
    print('Dataset found on Drive — copying to /content/ ...')
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    shutil.copytree(DRIVE_DATA, DATA_DIR)
    print('Done ✓')
else:
    print('Dataset not on Drive — downloading from Kaggle (one-time, ~10 min)...')
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    uploaded_name = list(uploaded.keys())[0]
    creds = json.loads(uploaded[uploaded_name])
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(creds, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print(f'Kaggle credentials configured (user: {creds["username"]}).')
    download_dataset(DATA_DIR)
    print('Saving to Google Drive for future sessions...')
    shutil.copytree(DATA_DIR, DRIVE_DATA)
    print('Saved to Drive ✓ — future sessions will skip the Kaggle download')

from data_loader import load_metadata
from preprocessing import make_splits, make_dataloaders, compute_class_weights

df            = load_metadata(DATA_DIR)
train_df, val_df, test_df = make_splits(df)
loaders       = make_dataloaders(train_df, val_df, test_df, batch_size=32, num_workers=2)
class_weights = compute_class_weights(train_df)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Class weights: {[round(w, 4) for w in class_weights.tolist()]}")

## Cell 5 — Train ResNet18

**Architecture:** ImageNet ResNet18 → replace `Linear(512→1000)` with `Linear(512→7)`  
**Parameters:** ~11M total (all trainable)  
**LR:** backbone=1e-4, head=1e-3 (two-speed Adam)  
⏱ **Expected:** ~1–2 min/epoch on T4 → 15–40 epochs total

In [ ]:
import os
from models.transfer_models import build_resnet18
from train import train

RESNET_CHECKPOINT = os.path.join(OUT_DIR, "models", "resnet18_best.pt")

resnet_model, resnet_param_groups = build_resnet18(num_classes=7, pretrained=True)

# Count params
total = sum(p.numel() for p in resnet_model.parameters())
print(f"ResNet18 total parameters: {total:,} (~{total/1e6:.1f}M)")
print(f"Training ResNet18...\n")

resnet_history = train(
    model          = resnet_model,
    train_loader   = loaders["train"],
    val_loader     = loaders["val"],
    class_weights  = class_weights,
    device         = device,
    num_epochs     = 30,
    lr             = 1e-3,          # fallback lr — overridden by param_groups
    weight_decay   = 1e-4,
    early_stopping_patience = 7,
    checkpoint_path = RESNET_CHECKPOINT,
    param_groups   = resnet_param_groups,
)

print(f"\nResNet18 checkpoint saved to: {RESNET_CHECKPOINT}")

## Cell 6 — Train EfficientNet-B0

**Architecture:** ImageNet EfficientNet-B0 → replace `Linear(1280→1000)` with `Linear(1280→7)`  
**Parameters:** ~5.3M total (all trainable)  
**LR:** backbone=1e-4, head=1e-3 (two-speed Adam)  
⏱ **Expected:** ~1–2 min/epoch on T4 → 15–40 epochs total

In [ ]:
import os
from models.transfer_models import build_efficientnet_b0
from train import train

EFFNET_CHECKPOINT = os.path.join(OUT_DIR, "models", "efficientnet_b0_best.pt")

effnet_model, effnet_param_groups = build_efficientnet_b0(num_classes=7, pretrained=True)

total = sum(p.numel() for p in effnet_model.parameters())
print(f"EfficientNet-B0 total parameters: {total:,} (~{total/1e6:.1f}M)")
print(f"Training EfficientNet-B0...\n")

effnet_history = train(
    model          = effnet_model,
    train_loader   = loaders["train"],
    val_loader     = loaders["val"],
    class_weights  = class_weights,
    device         = device,
    num_epochs     = 30,
    lr             = 1e-3,
    weight_decay   = 1e-4,
    early_stopping_patience = 7,
    checkpoint_path = EFFNET_CHECKPOINT,
    param_groups   = effnet_param_groups,
)

print(f"\nEfficientNet-B0 checkpoint saved to: {EFFNET_CHECKPOINT}")

## Cell 7 — Plot learning curves (both models)

In [ ]:
import os
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Transfer Learning — Learning Curves", fontsize=13)

for row, (name, history) in enumerate([
    ("ResNet18",       resnet_history),
    ("EfficientNet-B0", effnet_history),
]):
    axes[row, 0].plot(history["train_loss"], label="Train")
    axes[row, 0].plot(history["val_loss"],   label="Val")
    axes[row, 0].set_title(f"{name} — Loss")
    axes[row, 0].set_xlabel("Epoch")
    axes[row, 0].legend()

    axes[row, 1].plot([a*100 for a in history["train_acc"]], label="Train")
    axes[row, 1].plot([a*100 for a in history["val_acc"]],   label="Val")
    axes[row, 1].set_title(f"{name} — Accuracy")
    axes[row, 1].set_xlabel("Epoch")
    axes[row, 1].set_ylabel("%")
    axes[row, 1].legend()

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "figures", "transfer_learning_curves.png")
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")

## Cell 8 — Evaluate ResNet18 on test set

In [ ]:
from evaluate import evaluate_model, print_results, plot_confusion_matrix

# Load best checkpoint
resnet_model.load_state_dict(torch.load(RESNET_CHECKPOINT, map_location=device))
resnet_model = resnet_model.to(device)

print("=" * 60)
print("RESNET18 — TEST SET EVALUATION")
print("=" * 60)
resnet_results = evaluate_model(resnet_model, loaders["test"], device)
print_results(resnet_results)

plot_confusion_matrix(
    resnet_results["confusion_matrix"],
    save_path=os.path.join(OUT_DIR, "figures", "resnet18_confusion_matrix.png")
)

## Cell 9 — Evaluate EfficientNet-B0 on test set

In [ ]:
# Load best checkpoint
effnet_model.load_state_dict(torch.load(EFFNET_CHECKPOINT, map_location=device))
effnet_model = effnet_model.to(device)

print("=" * 60)
print("EFFICIENTNET-B0 — TEST SET EVALUATION")
print("=" * 60)
effnet_results = evaluate_model(effnet_model, loaders["test"], device)
print_results(effnet_results)

plot_confusion_matrix(
    effnet_results["confusion_matrix"],
    save_path=os.path.join(OUT_DIR, "figures", "efficientnet_b0_confusion_matrix.png")
)

## Cell 10 — Combined comparison table

The core result of the project: baseline CNN vs. ResNet18 vs. EfficientNet-B0.
Load the baseline results from its checkpoint for a fair comparison.

In [ ]:
import os
import pandas as pd
from models.baseline_cnn import build_baseline_cnn
from evaluate import evaluate_model

BASELINE_CHECKPOINT = os.path.join(OUT_DIR, "models", "baseline_cnn_best.pt")

# Load baseline model and evaluate
baseline_model = build_baseline_cnn(num_classes=7)
if os.path.exists(BASELINE_CHECKPOINT):
    baseline_model.load_state_dict(torch.load(BASELINE_CHECKPOINT, map_location=device))
    baseline_model = baseline_model.to(device)
    baseline_results = evaluate_model(baseline_model, loaders["test"], device)
    baseline_loaded = True
    print("Baseline checkpoint loaded ✓")
else:
    baseline_loaded = False
    print("⚠️  Baseline checkpoint not found — run Stage 3 first to generate it.")

# Build comparison table
def make_row(name, results):
    return {
        "Model"           : name,
        "Accuracy"        : f"{results['accuracy']*100:.2f}%",
        "Macro F1"        : f"{results['macro_f1']:.4f}",
        "Weighted F1"     : f"{results['weighted_f1']:.4f}",
        "Recall — mel"    : f"{results['malignant_recall']['mel']:.4f}",
        "Recall — bcc"    : f"{results['malignant_recall']['bcc']:.4f}",
        "Recall — akiec"  : f"{results['malignant_recall']['akiec']:.4f}",
    }

rows = []
if baseline_loaded:
    rows.append(make_row("Baseline CNN (scratch)", baseline_results))
rows.append(make_row("ResNet18 (fine-tuned)",       resnet_results))
rows.append(make_row("EfficientNet-B0 (fine-tuned)", effnet_results))

comparison_df = pd.DataFrame(rows)

print("\n" + "=" * 75)
print("MODEL COMPARISON — HAM10000 TEST SET")
print("=" * 75)
print(comparison_df.to_string(index=False))
print("=" * 75)
print("\nNote: Recall on mel/bcc/akiec is the primary clinical metric.")
print("High accuracy with low macro-F1 = model predicting majority class (nv).")

# Save to CSV
csv_path = os.path.join(OUT_DIR, "model_comparison.csv")
comparison_df.to_csv(csv_path, index=False)
print(f"\nSaved → {csv_path}")

## Cell 11 — Sanity check: flag if transfer model is worse than baseline

In [ ]:
if baseline_loaded:
    print("Macro F1 comparison:")
    print(f"  Baseline CNN  : {baseline_results['macro_f1']:.4f}")
    print(f"  ResNet18      : {resnet_results['macro_f1']:.4f}")
    print(f"  EfficientNet  : {effnet_results['macro_f1']:.4f}")
    print()

    for name, res in [("ResNet18", resnet_results), ("EfficientNet-B0", effnet_results)]:
        if res["macro_f1"] < baseline_results["macro_f1"]:
            print(f"⚠️  WARNING: {name} macro-F1 ({res['macro_f1']:.4f}) is LOWER than "
                  f"baseline ({baseline_results['macro_f1']:.4f}).")
            print(f"   This is unusual. Possible causes:")
            print(f"   1. Transfer model trained for fewer epochs — try increasing num_epochs or patience")
            print(f"   2. LR too high for backbone — try reducing backbone LR to 1e-5")
            print(f"   3. Overfitting — check learning curves for diverging train/val loss")
        else:
            print(f"✓  {name} outperforms baseline (macro-F1: {res['macro_f1']:.4f} vs {baseline_results['macro_f1']:.4f})")
else:
    print("Baseline checkpoint missing — run Stage 3 to enable comparison.")

## Stage 4 complete ✓

**Before moving to Stage 5 (Grad-CAM), verify all boxes below:**

- [ ] Both ResNet18 and EfficientNet-B0 trained without errors
- [ ] Both checkpoints saved: `resnet18_best.pt`, `efficientnet_b0_best.pt`
- [ ] Comparison table printed — all three models present
- [ ] Transfer models have **higher macro-F1 than baseline CNN** — if not, see Cell 11 warning
- [ ] Malignant-class recall (mel, bcc, akiec) is higher for transfer models than baseline
- [ ] `model_comparison.csv` saved to `outputs/`
- [ ] No model shows accuracy >90% with macro-F1 <0.50 (majority-class collapse)

**Share the comparison table before continuing to Stage 5.**

---

### Design decisions summary

| Decision | Choice | Reason |
|---|---|---|
| Freeze vs fine-tune | Fine-tune end-to-end | Dermoscopic images differ from ImageNet; backbone must adapt |
| Backbone LR | 1e-4 (10× lower) | Prevents catastrophic forgetting of pretrained features |
| Head LR | 1e-3 | New linear layer needs aggressive updates from random init |
| ResNet18 choice | ~11M params, well-cited in medical imaging | Simple, fast, explainable baseline transfer model |
| EfficientNet-B0 choice | ~5.3M params, compound scaling | Tests if efficiency-optimised architecture helps on small dataset |
| max epochs | 30 (vs 50 for baseline) | Transfer models converge faster — pretrained features give a head start |
| Same train/val/test splits | yes (random_state=42) | Required for fair comparison — identical data for all three models |

---

### Likely interview questions on this stage

**Q1: Why does transfer learning help here, given that ImageNet and dermoscopy look nothing alike?**  
Transfer learning helps because the low-level features learned on ImageNet (edges, colour gradients, textures, shapes) are still useful for dermoscopy — the backbone doesn't need to relearn how to detect edges from scratch. What changes is the higher-level combination of those features. With only ~7k training images, having pre-trained low-level detectors frees the model to use its capacity for the lesion-specific patterns instead of relearning basics.

**Q2: Why fine-tune end-to-end instead of freezing the backbone?**  
Dermoscopic images are photographed with polarised light through a dermatoscope — the texture and colour statistics are genuinely different from natural photos. A frozen ImageNet backbone would apply features calibrated for natural image statistics to a domain where those statistics don't hold. Fine-tuning allows the backbone to adapt. We use a 10× lower LR for the backbone to adapt gradually without destroying the learned representations (catastrophic forgetting).

**Q3: Why ResNet18 and EfficientNet-B0 specifically?**  
ResNet18 is the smallest ResNet with skip connections — it's widely cited in medical imaging literature as a transfer learning baseline, easy to explain, and fast to train on a free-tier GPU. EfficientNet-B0 uses compound scaling (balancing depth, width, resolution simultaneously) and achieves better accuracy-per-parameter than ResNet at this scale. Comparing them tests whether architectural efficiency translates to better performance on a small, imbalanced medical dataset.